# 第十三章：出版级可视化

**scRNA-seq 实践教程系列** | 基于 GSE136103 肝硬化数据集

---

## 写在前面

恭喜你走到了最后一篇。

过去 12 篇，我们从 GEO 下载原始数据开始，经历了 QC、归一化、降维、聚类、批次矫正、细胞注释、差异表达、功能富集、细胞通讯、转录因子调控、细胞组成、轨迹分析、共表达模块——一步步把 58,735 个细胞变成了一个关于肝硬化的生物学故事。

但分析只完成了一半。 另一半是展示。一个好的分析如果没有清晰的可视化和严谨的汇总，审稿人看不懂、读者记不住。

这最后一篇，我们做两件事：

制作出版级图表——多面板 UMAP、高级 marker 可视化、疾病特征基因展示
全系列总结——从 58,735 个细胞中我们学到了什么、和原文有什么异同、这个故事的下一步在哪里

## Part A：多面板 UMAP 可视化

### 为什么需要多面板

一张 UMAP 只能展示一个维度的信息。但审稿人和读者需要同时看到：细胞类型分布、疾病条件差异、供体来源——这就需要多面板并列。

### 三面板总览图

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import pandas as pd

# 全局画图设置（出版级）
mpl.rcParams["font.family"] = "Arial"
mpl.rcParams["font.size"] = 8
mpl.rcParams["axes.linewidth"] = 0.8
mpl.rcParams["xtick.major.width"] = 0.8
mpl.rcParams["ytick.major.width"] = 0.8

adata = sc.read_h5ad("results/04_annotated.h5ad")
print(f"数据: {adata.n_obs} cells × "
      f"{adata.n_vars} genes")

**输出：**

> 📊 输出：
> 数据: 58735 cells × 25354 genes

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel A: Cell type
sc.pl.umap(
    adata, color="cell_type",
    title="Cell Type", frameon=True,
    ax=axes[0], show=False
)

# Panel B: Condition
sc.pl.umap(
    adata, color="condition",
    palette={"Healthy": "#2ca02c",
             "Cirrhotic": "#d62728"},
    title="Condition", frameon=True,
    ax=axes[1], show=False
)

# Panel C: Donor
sc.pl.umap(
    adata, color="donor",
    title="Donor", frameon=True,
    ax=axes[2], show=False
)

for i, label in enumerate(["A", "B", "C"]):
    axes[i].text(-0.05, 1.05, label,
        transform=axes[i].transAxes,
        fontsize=14, fontweight="bold")

plt.tight_layout()
plt.savefig(
    "results/figures/12_umap_overview.png",
    dpi=300, bbox_inches="tight"
)
plt.close()

图 1：全景 UMAP——58,735 个细胞的细胞类型分布

💡 出版技巧：面板标签

图 2：精细注释 UMAP

Nature/Cell 系列期刊要求每个面板用粗体字母（A, B, C...）标记。用 ax.text() + fontweight="bold" 实现。坐标 (-0.05, 1.05) 把标签放在面板左上角外侧。

### 按条件拆分 UMAP

把 Healthy 和 Cirrhotic 并列展示，最能直观看出哪些细胞群在疾病中扩张或消失：

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, cond in zip(axes, ["Healthy", "Cirrhotic"]):
    mask = adata.obs["condition"] == cond
    n = mask.sum()

    # 先画灰色背景（所有细胞）
    sc.pl.umap(adata, color="cell_type",
               ax=ax, show=False,
               na_color="lightgrey", alpha=0.1)
    # 再画该条件的细胞
    sc.pl.umap(adata[mask], color="cell_type",
               ax=ax, show=False,
               title=f"{cond} (n={n:,})")

plt.tight_layout()
plt.savefig(
    "results/figures/12_umap_split_condition.png",
    dpi=300, bbox_inches="tight"
)
plt.close()

图 3：按条件拆分的 UMAP——Healthy vs Cirrhotic

在拆分图中，几个生物学特征一目了然：

Cirrhotic 中 HSCs/Mesenchyme 区域更密集——纤维化相关细胞扩张
Healthy 中特定巨噬细胞区域更突出——Kupffer 细胞在疾病中减少
Cirrhotic 中部分 T 细胞区域出现偏移——可能反映了疾病相关的 T 细胞状态变化

## Part B：Marker 可视化——Dotplot、Matrixplot、Stacked Violin

### 为什么要多种可视化

Dotplot、Matrixplot、Stacked Violin 看的是同样的数据（marker 基因 × 细胞类型），但侧重点不同：

图类型
展示信息
最适合场景

Dotplot
表达量 + 检测率
展示 marker 特异性（Nature 最爱）

Matrixplot
平均表达量热图
紧凑展示大量基因

Stacked Violin
表达分布形状
展示表达的异质性

### 定义 Marker 基因

我们为每种细胞类型准备了 canonical markers，分成 13 个类别：

In [ ]:
marker_categories = {
    "T cell": ["CD3D", "CD3E", "TRAC"],
    "CD4 T": ["CD4", "IL7R", "TCF7"],
    "CD8 T": ["CD8A", "CD8B", "GZMK"],
    "NK": ["NKG7", "KLRD1", "GNLY", "KLRF1"],
    "B cell": ["CD79A", "MS4A1", "CD19"],
    "Plasma": ["JCHAIN", "MZB1", "SDC1"],
    "Monocyte": ["CD14", "S100A8", "S100A9"],
    "Macrophage": ["CD68", "C1QA", "MARCO"],
    "cDC": ["FCER1A", "CD1C", "CLEC10A"],
    "pDC": ["LILRA4", "IRF7", "CLEC4C"],
    "Hepatocyte": ["ALB", "APOB", "SERPINA1"],
    "Endothelial": ["PECAM1", "CDH5", "VWF"],
    "HSC/Mesen.": ["ACTA2", "COL1A1", "DCN"],
}

### Dotplot（主力图）

In [ ]:
# 构建有序基因列表
all_markers = {}
for cat, genes in marker_categories.items():
    valid = [g for g in genes
             if g in adata.raw.var_names]
    if valid:
        all_markers[cat] = valid

sc.pl.dotplot(
    adata, var_names=all_markers,
    groupby="cell_type",
    standard_scale="var",
    use_raw=True,
    dendrogram=True,
    show=False
)
plt.savefig(
    "results/figures/12_dotplot_markers.png",
    dpi=300, bbox_inches="tight"
)
plt.close()

图 4：经典标记基因气泡图

Dotplot 解读要点：

点的大小 = 检测率（在多少比例的细胞中检测到该基因）
点的颜色 = 平均表达量（标准化后）
一个好的 marker 应该"大且深"在对应细胞类型，"小且浅"在其他类型
ALB 在 Hepatocytes 中应该是最大最深的点；CD3D 在 T cells 中最突出

### Matrixplot（紧凑热图）

In [ ]:
sc.pl.matrixplot(
    adata, var_names=all_markers,
    groupby="cell_type",
    standard_scale="var",
    use_raw=True,
    dendrogram=True,
    cmap="Blues",
    show=False
)
plt.savefig(
    "results/figures/12_matrixplot_markers.png",
    dpi=300, bbox_inches="tight"
)
plt.close()

图 5：标记基因矩阵图

Matrixplot 比 dotplot 更紧凑——用色块代替了圆点，省去了点大小这个维度。当你需要展示很多基因时（比如 50+ 个 marker），matrixplot 更合适。

### Stacked Violin（表达分布）

In [ ]:
# 取每类的前 2 个 marker 做 stacked violin
top2_markers = []
for genes in all_markers.values():
    top2_markers.extend(genes[:2])

sc.pl.stacked_violin(
    adata, var_names=top2_markers,
    groupby="cell_type",
    use_raw=True,
    dendrogram=True,
    swap_axes=True,
    show=False
)
plt.savefig(
    "results/figures/12_stacked_violin.png",
    dpi=300, bbox_inches="tight"
)
plt.close()

图 6：堆叠小提琴图

Stacked violin 的独特价值在于展示分布的形状——你能看到一个 marker 是在所有细胞中均匀表达（宽分布），还是只在一小部分细胞中高表达（双峰分布）。

⚠️ 踩坑预警：图太挤看不清

12 个细胞类型 × 30+ 个基因的 stacked violin 可能会非常拥挤。解决方案：

减少基因数量（每类只取 top 2）
用 swap_axes=True 让细胞类型在 Y 轴、基因在 X 轴
增加 figsize
或者直接改用 dotplot——大多数审稿人更习惯看 dotplot

## Part C：疾病特征基因 Feature Plot

### 单基因 UMAP 着色

Feature plot 把单个基因的表达量映射到 UMAP 上。对于展示"这个基因在哪种细胞中表达"非常直观：

In [ ]:
disease_genes = [
    "COL1A1", "ACTA2", "TGFB1",  # 纤维化
    "TNF", "IL1B",                 # 炎症
    "ALB",                         # 肝功能
    "CD68", "TREM2",               # 巨噬细胞
    "NKG7", "GZMB",                # 免疫杀伤
    "PECAM1",                      # 内皮
    "PDCD1",                       # T 细胞耗竭
]
# 过滤存在的基因
disease_genes = [g for g in disease_genes
                 if g in adata.raw.var_names]

fig, axes = plt.subplots(3, 4, figsize=(20, 15))
for ax, gene in zip(axes.flat, disease_genes):
    sc.pl.umap(
        adata, color=gene,
        use_raw=True,
        cmap="Reds", vmin=0,
        title=gene, frameon=True,
        ax=ax, show=False
    )

# 隐藏多余的子图
for i in range(len(disease_genes), 12):
    axes.flat[i].set_visible(False)

plt.tight_layout()
plt.savefig(
    "results/figures/12_feature_plots.png",
    dpi=300, bbox_inches="tight"
)
plt.close()

图 7：疾病关键基因在 UMAP 上的表达分布

Feature plot 解读：

COL1A1：热点在 HSCs/Mesenchyme 区域，纤维化的直接标志
ACTA2：和 COL1A1 高度重叠——活化的肌成纤维细胞
TGFB1：在多种细胞中广泛表达，但巨噬细胞和 HSCs 中更高
ALB：精确定位在 Hepatocyte 群——肝细胞的经典标志
CD68：巨噬细胞标志，精准覆盖 Macrophage 区域
TREM2：SAMac 的关键标志——在巨噬细胞区域的一部分富集
NKG7：NK 细胞和部分 CD8+ T 细胞的细胞毒性标志

💡 配色建议

Feature plot 推荐用单色渐变色图：

"Reds" 或 "OrRd"：最常用，Nature/Cell 系列的标配
"Purples"：如果需要和 Reds 区分
避免 "jet" 或 "rainbow"——它们有视觉偏差（perceptual non-uniformity），在灰度打印时也不可读
设置 vmin=0 确保零表达显示为白色/浅色

## Part D：数据汇总统计

### 全数据集概览表

在论文的方法部分和补充材料中，你需要一张清晰的数据汇总表：

In [ ]:
print("=" * 50)
print("📊 数据集汇总统计")
print("=" * 50)
print(f"总细胞数: {adata.n_obs:,}")
print(f"总基因数: {adata.n_vars:,}")
print(f"供体数: {adata.obs['donor'].nunique()}")
print(f"大类细胞类型: "
      f"{adata.obs['cell_type'].nunique()}")
print(f"细分亚群: "
      f"{adata.obs['cell_subtype'].nunique()}")
print()
print("--- 按条件 ---")
for cond in ["Healthy", "Cirrhotic"]:
    n = (adata.obs["condition"] == cond).sum()
    print(f"  {cond}: {n:,} cells")

**输出：**

> 📊 输出：
> ==================================================
> 📊 数据集汇总统计
> ==================================================
> 总细胞数: 58,735
> 总基因数: 25,354
> 供体数: 10
> 大类细胞类型: 12
> 细分亚群: 26
> 
> --- 按条件 ---
>   Healthy: 34,095 cells
>   Cirrhotic: 24,640 cells

### 细胞类型分布表

In [ ]:
ct_summary = adata.obs.groupby(
    ["cell_type", "condition"]
).size().unstack(fill_value=0)
ct_summary["Total"] = ct_summary.sum(axis=1)
ct_summary = ct_summary.sort_values(
    "Total", ascending=False
)
ct_summary["Pct"] = (
    ct_summary["Total"] / adata.n_obs * 100
).round(1)
print(ct_summary.to_string())

**输出：**

> 📊 输出：
>                    Healthy  Cirrhotic  Total   Pct
> T cells              12180       8229  20409  34.7
> NK cells              6502       4304  10806  18.4
> Endothelial           4891       2994   7885  13.4
> Monocytes             2318       2306   4624   7.9
> Macrophages           1547       2102   3649   6.2
> Hepatocytes           2012       1018   3030   5.2
> HSCs/Mesenchyme        987       1273   2260   3.8
> cDC                   1298        834   2132   3.6
> B cells               1154        768   1922   3.3
> Proliferating          412        377    789   1.3
> Plasma cells           398        263    661   1.1
> pDC                    396        172    568   1.0

关键观察：

T 细胞占比最大（34.7%），和肝脏作为免疫器官的特性一致
Macrophages 在 Cirrhotic 中绝对数量增加（1,547 → 2,102）——尽管总细胞数在减少
Hepatocytes 在 Cirrhotic 中大幅减少（2,012 → 1,018）——反映了肝硬化的实质细胞损失
HSCs/Mesenchyme 在 Cirrhotic 中比例增加——纤维化细胞扩张的直接证据

### 供体分布

In [ ]:
donor_summary = adata.obs.groupby(
    ["donor", "condition"]
).size().reset_index(name="n_cells")
for _, row in donor_summary.iterrows():
    print(f"  {row['donor']} ({row['condition']}): "
          f"{row['n_cells']:,} cells")

**输出：**

> 📊 输出：
>   Donor1 (Healthy): 4,120 cells
>   Donor2 (Healthy): 5,891 cells
>   Donor3 (Healthy): 7,203 cells
>   Donor4 (Healthy): 8,456 cells
>   Donor5 (Healthy): 8,425 cells
>   Donor6 (Cirrhotic): 3,892 cells
>   Donor7 (Cirrhotic): 4,567 cells
>   Donor8 (Cirrhotic): 5,234 cells
>   Donor9 (Cirrhotic): 5,789 cells
>   Donor10 (Cirrhotic): 5,158 cells

💡 汇总表用途

这些数字在论文中会出现在三个地方：

Abstract："We analyzed 58,735 cells from 10 donors..."
Methods → Data processing：完整的过滤统计
Supplementary Table 1：按供体、条件、细胞类型的完整交叉表

## Part E：出版级图表制作指南

### 期刊图表规范

不同期刊对图表有严格的格式要求。以下是主流期刊的核心参数：

In [ ]:
# Nature 系列的图表规范
nature_params = {
    "单栏宽度": "89 mm (≈3.5 in)",
    "双栏宽度": "183 mm (≈7.2 in)",
    "字体": "Arial / Helvetica",
    "字号": "5-8 pt（最小 5 pt）",
    "分辨率": "≥300 dpi（位图）",
    "格式": "PDF / EPS（矢量）> TIFF > PNG",
    "颜色模式": "RGB（在线）或 CMYK（印刷）",
}

# Cell 系列
cell_params = {
    "单栏宽度": "85 mm",
    "双栏宽度": "178 mm",
    "字体": "Arial / Helvetica",
    "字号": "6-8 pt",
    "分辨率": "≥300 dpi",
}

### 配色方案

⚠️ 踩坑预警：默认配色不适合发表

Scanpy 的默认 UMAP 配色虽然好看，但有几个问题：

色弱不友好——红绿对比在色盲患者中无法区分
灰度打印不可读——很多审稿人会打印黑白版
类别太多时颜色会重复——12 种细胞类型已经接近极限

In [ ]:
# 推荐配色方案

# 方案 1：基于 Tableau 20（最多 20 个类别）
from matplotlib.colors import ListedColormap
tableau20 = [
    "#1f77b4", "#aec7e8", "#ff7f0e", "#ffbb78",
    "#2ca02c", "#98df8a", "#d62728", "#ff9896",
    "#9467bd", "#c5b0d5", "#8c564b", "#c49c94",
]

# 方案 2：疾病条件对比（直觉色）
condition_colors = {
    "Healthy": "#2ca02c",   # 绿 = 健康
    "Cirrhotic": "#d62728", # 红 = 疾病
}

# 方案 3：连续色图推荐
# 表达量：'Reds', 'YlOrRd', 'viridis'
# 差异值（有正负）：'RdBu_r', 'coolwarm'
# 富集值：'YlGnBu'

### 字体与排版

In [ ]:
# 出版级全局设置模板
def set_publication_style():
    """设置适合 Nature/Cell 的画图参数"""
    mpl.rcParams.update({
        "font.family": "Arial",
        "font.size": 7,
        "axes.titlesize": 8,
        "axes.labelsize": 7,
        "xtick.labelsize": 6,
        "ytick.labelsize": 6,
        "legend.fontsize": 6,
        "axes.linewidth": 0.5,
        "xtick.major.width": 0.5,
        "ytick.major.width": 0.5,
        "xtick.major.size": 3,
        "ytick.major.size": 3,
        "figure.dpi": 300,
        "savefig.dpi": 300,
        "savefig.bbox": "tight",
        "savefig.pad_inches": 0.05,
        "pdf.fonttype": 42,  # 嵌入字体
        "ps.fonttype": 42,
    })

set_publication_style()

💡 关键参数解释

pdf.fonttype = 42：这个很重要！它让 PDF 中的字体以 TrueType 嵌入，而不是转成路径。审稿系统和排版软件都需要可编辑的字体。
字号 7pt 是 Nature 的黄金字号——够小但还能看清。
axes.linewidth = 0.5：比 Matplotlib 默认的 0.8 更精致。

### 保存格式建议

In [ ]:
# 矢量图（推荐用于主图）
plt.savefig("figure1.pdf", format="pdf")
# 高分辨率位图（用于补充材料或嵌入 PPT）
plt.savefig("figure1.png", dpi=300)
# TIFF（某些期刊要求）
plt.savefig("figure1.tiff", dpi=300,
            pil_kwargs={"compression": "tiff_lzw"})

## 🎓 全系列回顾：从 58,735 个细胞到一个完整的故事

### 我们做了什么

回顾一下 13 篇的完整旅程：

篇章
主题
核心产出

01
数据加载
10 供体 × 20 文库 → 合并 AnnData

02
QC + Doublet
MAD 自适应过滤 + Scrublet → 58,735 cells

03
归一化/降维
LogNormalize → 3000 HVG → PCA → UMAP

04
批次矫正
Harmony 整合 20 个文库

05
细胞注释
手动 + CellTypist → 12 大类、26 亚群

06
差异表达
Wilcoxon + Pseudobulk → 数千个 DEGs

07
功能富集
GO/KEGG ORA + GSEA → 通路变化

08
细胞通讯
LIANA → 配体-受体互作网络

09
TF 调控
CollecTRI + decoupler → 关键转录因子

10
细胞组成
比例检验 → 组成变化定量化

11
轨迹分析
PAGA + DPT → 分化轨迹和拟时序

12
共表达/评分
共表达模块 + 基因集打分

13
可视化/汇总
出版级图表 + 全故事整合

### 核心生物学发现

从 58,735 个细胞的分析中，我们描绘出了一幅肝硬化重塑细胞图谱的全景画面：

🔬 发现 1：Kupffer 细胞耗竭与 SAMac 崛起

这是本数据集最核心的发现。在健康肝脏中，组织驻留的 Kupffer 细胞（MARCO+TIMD4+CD163+）是主要的巨噬细胞群体。但在肝硬化中：

Kupffer 细胞比例显著下降（组成分析，第 10 篇）
取而代之的是一群新的 TREM2+CD9+SPP1+ 巨噬细胞——即 SAMac
共表达分析（第 12 篇）发现了 Kupffer 模块下调 + SAMac 模块上调
细胞通讯分析（第 8 篇）揭示了 SAMac 通过 SPP1-CD44 轴与星状细胞互作

这说明肝硬化中的巨噬细胞群体经历了一次"换班"——从维持稳态的哨兵变成了促纤维化的推手。

🔬 发现 2：星状细胞活化与纤维化网络

HSCs/Mesenchyme 在肝硬化中：

比例增加（第 10 篇）
Fibrosis 基因集评分显著升高（第 12 篇）
ACTA2、COL1A1、COL3A1 大幅上调（第 6 篇）
TGF-β 通路相关 TF 活性增强（第 9 篇）
接收来自 SAMac 的 TGFB1 信号（第 8 篇）

这形成了一个完整的"纤维化回路"：SAMac 分泌 TGFB1 → 激活 HSC → HSC 分泌胶原 → 组织纤维化。

🔬 发现 3：内皮细胞功能重塑

肝窦内皮细胞（LSEC）在肝硬化中：

CLEC4G、STAB2 等窗孔标志物下调（第 6 篇）——去窗孔化
Angiogenesis score 升高（第 12 篇）——异常血管新生
内皮亚群比例发生变化（第 10 篇）
VEGF 通路相关通讯活跃（第 8 篇）

去窗孔化导致肝窦功能障碍，进一步加重肝损伤。

🔬 发现 4：免疫微环境失调

T 细胞和 NK 细胞在肝硬化中：

T 细胞 Exhaustion score 升高（第 12 篇）——PDCD1、LAG3 上调
NK 细胞 Cytotoxicity score 轻度下降（第 12 篇）
促炎细胞因子基因（TNF、IL1B）在多种免疫细胞中上调（第 6 篇）
抗原呈递通路在 cDC 中增强（第 7 篇）

慢性炎症环境导致免疫细胞功能失调——既有过度激活的一面（炎症），也有功能耗竭的一面（T 细胞 exhaustion）。

### 与 Ramachandran et al. 原文的全面对照

📖 我们的结果和原文的异同：

发现
原文
我们的分析
一致性

SAMac 鉴定
TREM2+CD9+ SAMac
发现相同标志物模块
✅ 高度一致

Kupffer 耗竭
Kupffer 比例下降
组成分析 + 共表达验证
✅ 高度一致

SAMac-HSC 互作
SPP1-CD44、TGFB1 通路
LIANA 检测到相同配体-受体对
✅ 一致

内皮重塑
LSEC 去窗孔化
标志物下调 + 亚群比例变化
✅ 一致

纤维化 HSC 活化
COL1A1/ACTA2 上调
DEG + Fibrosis score 验证
✅ 一致

T 细胞耗竭
未深入分析
Exhaustion score 升高
🆕 扩展发现

肝细胞代谢下降
Figure 2 中展示
Lipid_Metabolism score ↓
✅ 一致

胆管上皮细胞
鉴定为独立群体
细胞数太少未独立标注
⚠️ 差异

总结： 我们的分析在核心发现上和原文高度一致——这验证了我们的分析流程是可靠的。同时，我们通过 T 细胞耗竭分析和共表达模块发现，补充了原文未深入探讨的免疫失调角度。胆管上皮细胞的缺失是一个已知的限制，可能与聚类 resolution 和细胞数量有关。

## 未来方向

### 空间转录组学

scRNA-seq 最大的局限是丢失了空间信息。细胞从组织中被消化分离后，我们不再知道哪些细胞是邻居、它们在组织中的哪个区域。

空间转录组学技术（Visium、MERFISH、Slide-seq）可以在保留空间位置的同时检测基因表达。想象一下：不仅知道 SAMac 和 HSC 之间有信号互作（LIANA 告诉我们的），还能看到它们在纤维化瘢痕区域物理上相邻——这将大大增强因果推断。

### 多组学整合

ATAC-seq（染色质可及性）+ RNA-seq 的多组学整合可以揭示调控层面的变化。比如，TREM2 在 SAMac 中上调——是因为它的启动子被打开了吗？哪些转录因子在结合它的增强子区域？这些问题需要单细胞多组学数据才能回答。

### 临床转化

SAMac 的发现已经催生了治疗靶点的探索。TREM2 靶向治疗在临床前研究中显示出减轻纤维化的潜力。从单细胞图谱到药物靶点，是转化医学最激动人心的方向之一。

### 机器学习与基础模型

scGPT、Geneformer 等单细胞基础模型（foundation models）正在改变分析范式。用预训练模型做细胞注释、基因扰动预测、虚拟筛选——这是 2024-2025 年的前沿方向。我们的数据集完全可以作为 fine-tuning 的素材。

## 给读者的建议

### 如果你是初学者

从头到尾跑一遍代码。 不要只读文章——亲手跑一遍，遇到报错，解决报错，这个过程比任何教程都有教育意义。
换一个数据集再做一遍。 肝脏 scRNA-seq 只是一个例子。找一个和你课题相关的数据集（GEO 上有大量公开数据），用同样的流程走一遍。
读原始论文。 我们的教程是"怎么做"，原始论文是"为什么做"和"发现了什么"。结合起来看收获最大。

### 如果你要发论文

方法部分写清楚每个参数。 不要只写 "we performed QC"——写清楚 MAD 阈值、过滤标准、HVG 数量、Harmony 参数、Leiden resolution。可重复性是科学的基石。
提供完整的代码。 把你的分析脚本放到 GitHub 上。审稿人和读者会感谢你。
关注图表质量。 审稿人的第一印象来自图表。花时间调好配色、字体、分辨率。
诚实报告局限性。 每种方法都有局限。承认它们比掩盖它们更能赢得审稿人的信任。

### 如果你要带学生

教 WHY 比教 HOW 更重要。 学生可以自己查函数文档，但理解"为什么要做批次矫正"需要你的指导。
鼓励犯错。 用错误的参数跑一遍、看看结果有多离谱——这比"正确地跑一遍"更有教育意义。
建立 checkpoint 系统。 每个大步骤保存 h5ad 文件。学生不需要从头跑全部流程，可以从任何一个 checkpoint 开始。

## 最后的话

单细胞转录组学让我们第一次以单个细胞为分辨率理解疾病。从 58,735 个细胞中，我们看到了肝硬化如何重塑每一种细胞类型——从组织驻留巨噬细胞的消亡到瘢痕相关巨噬细胞的崛起，从星状细胞的活化到肝细胞的功能衰退，从免疫细胞的过度激活到 T 细胞的功能耗竭。

这不仅仅是一个技术流程的演示。我们试图展示的是一种思维方式：如何从海量数据中提出问题、验证假设、发现规律、讲述故事。这种能力不会因为工具的更新换代而过时。

技术在变，但科学的逻辑不变。

感谢你陪我们走完这 13 篇。祝你在自己的数据中发现精彩的故事。

## 小结

这一篇（也是全系列的最后一篇）我们完成了：

✅ 多面板 UMAP 可视化（细胞类型 + 条件 + 供体）
✅ 按条件拆分 UMAP
✅ 三种 Marker 可视化（Dotplot、Matrixplot、Stacked Violin）
✅ 疾病特征基因 Feature Plot（12 个关键基因）
✅ 数据汇总统计表
✅ 出版级图表制作指南（Nature/Cell 规范）
✅ 全系列 13 篇回顾与生物学故事整合
✅ 与原文的全面对照
✅ 未来方向展望

关键数字（全系列汇总）：

总细胞数：58,735（Healthy 34,095 + Cirrhotic 24,640）
总基因数：25,354
供体数：10（5 Healthy + 5 Cirrhotic）
大类细胞类型：12
细分亚群：26
分析方法：13 种（QC → 可视化）

全系列输出文件：

results/01_merged.h5ad → 原始合并数据
results/02_clustered.h5ad → QC + 降维 + 聚类
results/03_integrated.h5ad → Harmony 整合
results/04_annotated.h5ad → 完整注释
results/deg_*.csv → 差异表达结果
results/enrichment_*.csv → 功能富集结果
results/liana_*.csv → 细胞通讯结果
results/tf_activities_*.csv → TF 活性结果
results/composition_*.csv → 组成分析结果
results/trajectory_*.csv → 轨迹分析结果
results/coexpression_modules_*.csv → 共表达模块
results/geneset_scores_condition.csv → 基因集评分
results/figures/ → 所有图表